In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np
from shapely import wkt
from sklearn.cluster import KMeans
import cartopy.crs as ccrs
import cartopy.feature as cfeature

raw_df = pd.read_csv("metadata_JPSS-1.csv")
df = raw_df[raw_df["KR_Sea_overlap"] > 0.0]
print(f"# of datas after filtering by cond 'raw_df['KR_Sea_overlap'] > 0.0' : {len(df)}")

In [ ]:
# Helper Functions

# HH:MM:SS → seconds since 00:00:00
def hhmmss_to_sec(s):
    if pd.isna(s):
        return None
    h, m, sec = s.split(":")
    return int(h) * 3600 + int(m) * 60 + int(sec)

def sec_to_hhmmss(x):
    h = int(x // 3600)
    m = int((x % 3600) // 60)
    s = int(x % 60)
    return f"{h:02d}:{m:02d}:{s:02d}"

def SecToUTC(x):
    return f"{int(x//3600):02d}:00"

def make_colors(data, color_map):
    bins = [0, 20, 40, 60, 80, 100]
    labels = ["0–20", "20–40", "40–60", "60–80", "80–100"]
    data["overlap_bin"] = pd.cut(
        data["overlap_mean"],
        bins=bins,
        labels=labels,
        include_lowest=True
    )
    colors = data["overlap_bin"].map(color_map)
    keys = list(color_map.keys())
    legend_elements = [Patch(facecolor=color_map[keys[i]], label=keys[i]) for i in range(len(keys))]
    return colors, legend_elements

def duration_boxplot(dur, explain):
    # 통계량 계산
    stats = {
        "N": dur.count(),
        "mean": dur.mean(),
        "std": dur.std(),
        "min": dur.min(),
        "median": dur.median(),
        "max": dur.max(),
    }

    # 통계 텍스트 문자열
    text_1 = (
        f"Total # = {stats['N']}\n"
        f"Mean = {stats['mean']:.3f}\n"
        f"Std = {stats['std']:.3f}"
    )

    text_2 = (
        f"Max = {stats['max']:.3f}\n"
        f"Median = {stats['median']:.3f}\n"
        f"Min = {stats['min']:.3f}"
    )

    text_3 = (
        f"# of (duration < 358 sec) : {len(dur[dur < 358])}"
    )

    plt.figure(figsize=(7,3))

    plt.boxplot(
        dur.dropna(),
        vert=False,
        showmeans=True,
        meanline=True,
        flierprops=dict(
            marker='o',
            markersize=6,
            markerfacecolor='red',
            markeredgecolor='black'
        )
    )

    # 텍스트 박스 생성
    plt.text(
        0.15, 0.90, text_1,
        transform=plt.gca().transAxes,
        ha="left",
        va="top",
        bbox=dict(
            boxstyle="round,pad=0.3",
            facecolor="white",
            edgecolor="black",
            alpha=0.9
        )
    )

    plt.text(
        0.4, 0.90, text_2,
        transform=plt.gca().transAxes,
        ha="left",
        va="top",
        bbox=dict(
            boxstyle="round,pad=0.3",
            facecolor="white",
            edgecolor="black",
            alpha=0.9
        )
    )

    plt.text(
        0.65, 0.9, text_3,
        transform=plt.gca().transAxes,
        ha="left",
        va="top",
        bbox=dict(
            boxstyle="round,pad=0.3",
            facecolor="white",
            edgecolor="black",
            alpha=0.9
        )
    )

    plt.xlabel("Scan duration (sec)")
    plt.title("Box plot of scan duration : [" + explain + "]")
    plt.tight_layout()
    plt.show()

In [ ]:
# 1. scan_start_HHMMSS의 통계량 및 시각화

df["scan_start_sec"] = df["scan_start_HHMMSS"].apply(hhmmss_to_sec)
df[["scan_start_HHMMSS", "scan_start_sec"]].head()

t = df["scan_start_sec"].dropna()

stats_t = {
    "N": t.count(),
    "mean": t.mean(),
    "std": t.std(),
    "min": t.min(),
    "median": t.median(),
    "max": t.max(),
}

text_t = (
    f"N = {stats_t['N']}\n"
    f"Mean = {sec_to_hhmmss(stats_t['mean'])}\n"
    #f"Std = {sec_to_hhmmss(stats_t['std'])} sec\n"
    f"Median = {sec_to_hhmmss(stats_t['median'])}\n"
    f"Min = {sec_to_hhmmss(stats_t['min'])}\n"
    f"Max = {sec_to_hhmmss(stats_t['max'])}"
)

plt.figure(figsize=(7,5))
plt.hist(t, bins=96)  # 24시간 / 15분 = 96 bins
plt.xlabel("Scan start time (UTC)")
plt.xlim(15*3600, 21*3600)
plt.ylabel("Count")
plt.title("Distribution of scan start time (UTC)\n(bin width = 15 minutes)")

plt.text(
    0.70, 0.88, text_t,
    transform=plt.gca().transAxes,
    ha="left",
    va="top",
    bbox=dict(
        boxstyle="round,pad=0.3",
        facecolor="white",
        edgecolor="black",
        alpha=0.9
    )
)

ax = plt.gca()
xticks = np.arange(15*3600, 21*3600 + 1, 3600)  # 2시간 간격
ax.set_xticks(xticks)
ax.set_xticklabels([SecToUTC(x) for x in xticks])

plt.show()

In [ ]:
# 1. (CONT)

counts = (
    df["scan_start_HHMMSS"]
    .dropna()
    .value_counts()
    .sort_index()
)

plt.figure(figsize=(5,10))

plt.barh(counts.index[::-1], counts.values[::-1], height = 0.5)

plt.ylabel("Scan start time (HH:MM:SS, UTC)")
plt.xlabel("Count")
plt.title("Distribution of scan_start_HH:MM:SS")

ax = plt.gca()
ax.xaxis.grid(
    True,
    linestyle="--",
    alpha=0.5
)

plt.yticks()
plt.tight_layout()
plt.show()

In [ ]:
# 2. scan_start_HHMM의 통계량 및 시각화
## df (contains only 'overlap' datas)

counts = (
    df["scan_start_HHMM"]
    .dropna()
    .value_counts()
    .sort_index()
)

plt.figure(figsize=(5,10))

plt.barh(counts.index[::-1], counts.values[::-1], height = 0.5)

plt.ylabel("Scan start time (HH:MM, UTC)")
plt.xlabel("Count")
plt.title("Distribution of scan_start_HH:MM")

ax = plt.gca()
ax.xaxis.grid(
    True,
    linestyle="--",
    alpha=0.5
)

plt.yticks()
plt.tight_layout()
plt.show()

In [ ]:
# 2. (CONT)
## raw_df

mean_by_hhmm = (
    raw_df.groupby("scan_start_HHMM", as_index=False)["KR_Sea_overlap"]
      .mean()
      .rename(columns={"KR_Sea_overlap": "overlap_mean"})
)

mean_by_hhmm.fillna(0, inplace=True)
mean_by_hhmm = mean_by_hhmm.sort_values("scan_start_HHMM")

color_map = {
    "0–20":   "#deebf7",
    "20–40":  "#c6dbef",
    "40–60":  "#9ecae1",
    "60–80":  "#6baed6",
    "80–100": "#2171b5",
}

colors, legend_elements = make_colors(mean_by_hhmm, color_map)

print(mean_by_hhmm)

plt.figure(figsize=(5,10))

plt.ylabel("Scan start time (HH:MM, UTC)")
plt.xlabel("overlap_mean (%)")
plt.title("Mean KR_EEZ Overlap by Scan Start Time (HH:MM)")

ax = plt.gca()

ax.xaxis.grid(
    True,
    linestyle="--",
    alpha=0.5
)

ax.legend(
    handles=legend_elements,
    title="Mean KR_EEZ overlap",
    loc="upper right",
    frameon=True
)

ax.barh(mean_by_hhmm["scan_start_HHMM"][::-1], mean_by_hhmm["overlap_mean"][::-1], height = 0.5, color=colors[::-1])

plt.yticks()
plt.tight_layout()
plt.show()

In [ ]:
# 3. scan duration의 통계량 및 시각화

import matplotlib.pyplot as plt

#df = raw_df[raw_df["duration [sec]"] >= 358]

dur = raw_df["duration [sec]"].astype(float)
overlap_dur = df["duration [sec]"].astype(float)

duration_boxplot(dur, "all data")
duration_boxplot(overlap_dur, "overlap > 0 % only")

In [ ]:
# 4. area_wkt를 활용한 촬영 영역 시각화

df = df.copy()
df["geometry"] = df["area_wkt"].apply(wkt.loads)

gdf = gpd.GeoDataFrame(
    df,
    geometry="geometry",
    crs="EPSG:4326"   # GRingPoint는 보통 lon/lat
)



gdf["center"] = (gdf["geometry"].representative_point())
centers = gpd.GeoSeries(gdf["center"], crs = gdf.crs)

fig, ax = plt.subplots(figsize=(12, 12))

gdf.plot(
    ax=ax,
    edgecolor="blue",
    facecolor="none",
    linewidth=0.8,
    alpha=0.3
)

centers.plot(
    ax=ax,
    marker="+",
    color="red",
    markersize=100,
    alpha=0.6
)

ax.set_title("Overlay of Satellite Footprints with Representative Points")
ax.set_aspect("equal")
plt.show()

print(f"# of Coordinate Flip occurance : {len(gdf[gdf['center'].x <= 50])}")

In [ ]:
# 4. KMeans 기반 클러스터링

gdf_m = gdf.to_crs("EPSG:5179")  # 미터 좌표(한국/동북아에 적합)
cent = gdf_m["center"].to_crs("EPSG:5179")
X = np.column_stack([cent.x.values, cent.y.values])

kmeans = KMeans(n_clusters=4, random_state=0, n_init="auto")
gdf["group"] = kmeans.fit_predict(X)
gdf = gdf[gdf['center'].x > 50] # ignore Coordinate Flip for clear visualization (since its # is only 1)

# 그룹별 색(취향대로)
colors = {0: "tab:purple", 1: "tab:purple", 2: "tab:purple", 3: "tab:purple",}

fig = plt.figure(figsize=(12, 8))
ax = plt.axes(projection=ccrs.PlateCarree())
# 배경 지도 요소
ax.add_feature(cfeature.LAND, alpha=0.4)
ax.add_feature(cfeature.OCEAN, alpha=0.3)
ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linewidth=0.5)
ax.gridlines(draw_labels=True, x_inline=False, y_inline=False)

# 시각화 범위 설정
ax.set_extent([85, 180, 5, 70], crs=ccrs.PlateCarree())


# footprint: 그룹별로 따로 plot
for k, c in colors.items():
    gdf[gdf["group"] == k].plot(
        ax=ax,
        edgecolor=c,
        facecolor="none",
        linewidth=0.8,
        alpha=0.35,
        label=f"Group {k}"
    )

# center (+): 그룹별로
for k, c in colors.items():
    centers = gdf[gdf["group"] == k]["center"]
    gpd.GeoSeries(centers, crs=gdf.crs).plot(
        ax=ax,
        marker="+",
        color="tab:red",
        markersize=80,
        linewidth=1.5,
        alpha=0.8
    )

ax.set_title("Overlay of Satellite Footprints with Representative Points")
ax.set_aspect("equal")
ax.set_xlabel("Longitude (°E)")
ax.set_ylabel("Latitude (°N)")
ax.legend()
ax.grid(True)
plt.show()

In [ ]:
# 4. (CONT)

# 문자열 -> time(시각)로 변환
gdf["scan_time"] = pd.to_datetime(gdf["scan_start_HHMMSS"], format="%H:%M:%S").dt.time #, errors="coerce").dt.time

t = pd.to_datetime(
    gdf["scan_start_HHMMSS"],
    format="%H:%M:%S",
    errors="coerce"
)

gdf["scan_sec"] = (
    t.dt.hour * 3600 +
    t.dt.minute * 60 +
    t.dt.second
)

summary = (
    gdf.dropna(subset=["scan_time"])
       .groupby("group")
       .agg(
           n=("scan_time", "size"),
           first=("scan_time", "min"),
           last=("scan_time", "max"),
           mean_time=("scan_sec", lambda s: sec_to_hhmmss(s.mean())),
           std_time=("scan_sec", lambda s: sec_to_hhmmss(s.std()))
       )
)

summary = summary.reset_index()
gdf["color"] = gdf["group"].map(colors)
summary["color"] = summary["group"].map(colors)

print(summary)

In [ ]:
# 5. Polygon.py에서 찾은 lon/lat 값이 min/max인 tif_name들을 이용해서 각 tif의 scan_start_HHMMSS 찾기

lon_max = "A2025249_1548_002.tif"
lat_min = "A2025245_1712_002.tif"
lat_max = "A2025225_1818_002.tif"
lon_min = "A2025005_1848_002.tif"

ls = [lon_min, lon_max, lat_min, lat_max]

for name in ls:
    i = df.loc[df['tif_name'] == name].index
    UTC = df.loc[i, "scan_start_HHMMSS"]
    print(f"{name} : {UTC.values}")